# MIND-small: abstract entity counts and title/abstract TF-IDF

Start from the CSVs produced by `notebooks/01_Data_Preprocessing.ipynb` (`data/processed/v1`). This notebook builds two article-level features and one set of user-history matching features for later model building:

| Feature | Source | Description | Data type |
|---|---|---|---|
| `num_abstract_entities` | `news.tsv` → abstract entity JSON | Number of identified entities in the abstract | Count (int32) |
| `tfidf_title_abstract` | `news.tsv` → title + abstract text | Sparse textual representation of article content | Numeric vector (sparse float32) |
| Entity-type matching (person / organisation / place) | `news.tsv` → entity `Type`; `behaviors.tsv` → user history | Type-specific article representations and separate history–candidate matching scores, with missing-type flags | Numeric scores (float32) + flags (bool) |

**Run:** run preprocessing first, configure paths below, then run top to bottom. Dependencies: pandas, NumPy, SciPy and scikit-learn. No click labels are used.

## 1. Feature definitions

**`num_abstract_entities`** counts the parsed annotation objects in `abstract_entities`. As with `num_title_entities`, this is the number of **annotation entries**, not mentions or occurrence offsets; one annotation with several offsets counts once. No confidence threshold is applied. Blank abstracts and empty entity lists give 0, so `metadata_missing` and `abstract_entities_missing` are kept alongside to distinguish a missing value from an observed zero.

**`tfidf_title_abstract`** is a TF-IDF vector over the concatenated text `title + " " + abstract`. The vocabulary and IDF weights are fitted on **`train/news.csv` only**, excluding `metadata_missing` rows, as required by the data dictionary. The same fitted vectorizer then transforms train, validation and test, so words that only appear in later partitions are ignored rather than leaking into the vocabulary.

Vectorizer settings (edit in Section 4 if needed):

- lowercase, English stop words removed, tokens of 2+ letters/digits
- unigrams and bigrams, `min_df=3`, `max_df=0.9`, at most 50,000 terms
- sublinear TF (`1 + log(tf)`) and L2-normalised rows, so long abstracts do not dominate

Each row is one `news_id`. The matrix is stored in sparse `.npz` format with a matching `news_ids.csv` row index; a dense CSV would be very large and mostly zeros.

## 2. Configure paths

Defaults follow the repository layout. Set `BT4222_PROJECT_ROOT`, `BT4222_PROCESSED_DIR` and/or `BT4222_TEXT_FEATURES_DIR` to override. Outputs go to `outputs/features/text_entity_v1`, which is git-ignored.

In [34]:
from pathlib import Path
import json
import os
import pickle

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
try:
    from IPython.display import display
except ImportError:  # Also permit running the code as a plain Python script.
    display = print

configured_root = os.environ.get('BT4222_PROJECT_ROOT')
PROJECT_ROOT = (
    Path(configured_root).expanduser().resolve()
    if configured_root else next(
        (p for p in [Path.cwd(), *Path.cwd().parents]
         if (p / 'src/data_loading.py').is_file()), None
    )
)

configured_processed = os.environ.get('BT4222_PROCESSED_DIR')
if configured_processed:
    PROCESSED_DIR = Path(configured_processed).expanduser().resolve()
elif PROJECT_ROOT is not None:
    PROCESSED_DIR = PROJECT_ROOT / 'data/processed/v1'
else:
    raise FileNotFoundError(
        'Open this notebook inside the cloned repository, or set '
        'BT4222_PROCESSED_DIR to the preprocessed version folder.'
    )

default_output = (
    PROJECT_ROOT / 'outputs/features/text_entity_v1'
    if PROJECT_ROOT is not None
    else PROCESSED_DIR.parent / 'text_entity_features_v1'
)
OUTPUT_DIR = Path(os.environ.get('BT4222_TEXT_FEATURES_DIR', str(default_output))).expanduser().resolve()

print('Processed data:', PROCESSED_DIR)
print('Output folder: ', OUTPUT_DIR)

Processed data: /Users/boogywumpy/Desktop/bt4222/data/processed/v1
Output folder:  /Users/boogywumpy/Desktop/bt4222/outputs/features/text_entity_v1


## 3. Read available partitions

Partitions are taken from the preprocessing manifest. CSVs are read as strings with `keep_default_na=False` so literal text such as `NA` stays text.

In [35]:
manifest_path = PROCESSED_DIR / 'manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'{manifest_path} is missing. Run 01_Data_Preprocessing.ipynb first.')
source_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
PARTITIONS = source_manifest['partitions']
if 'train' not in PARTITIONS:
    raise ValueError('A training partition is needed to fit the TF-IDF vocabulary.')

NEWS_COLUMNS = [
    'news_id', 'title', 'abstract', 'abstract_entities_parsed', 'num_abstract_entities',
    'metadata_missing', 'title_missing', 'abstract_missing', 'abstract_entities_missing',
]
FLAG_COLUMNS = ['metadata_missing', 'abstract_missing', 'abstract_entities_missing']


def load_news(partition):
    news = pd.read_csv(
        PROCESSED_DIR / partition / 'news.csv', usecols=NEWS_COLUMNS,
        dtype='string', keep_default_na=False, encoding='utf-8',
    )
    if not news['news_id'].is_unique:
        raise ValueError(f'{partition}: duplicate news_id would multiply downstream joins.')
    for column in NEWS_COLUMNS:
        if column.endswith('_missing'):
            if not news[column].isin(['True', 'False']).all():
                raise ValueError(f'{partition}: invalid boolean in {column}')
            news[column] = news[column].eq('True')
    return news


news_by_partition = {partition: load_news(partition) for partition in PARTITIONS}
for partition, news in news_by_partition.items():
    print(f'{partition}: {len(news):,} articles')

train: 43,370 articles
validation: 38,421 articles
test: 42,416 articles


## 4. `num_abstract_entities`

Recompute the count from the parsed abstract entity JSON and verify it against the value already supplied by preprocessing. A mismatch raises an error instead of silently changing the data.

In [36]:
def count_abstract_entities(value, news_id):
    try:
        parsed = json.loads(value)
    except (TypeError, json.JSONDecodeError) as exc:
        raise ValueError(f'{news_id}: invalid abstract_entities_parsed JSON') from exc
    if not isinstance(parsed, list) or not all(isinstance(e, dict) for e in parsed):
        raise ValueError(f'{news_id}: abstract_entities_parsed must be a list of objects')
    return len(parsed)


def derive_abstract_entity_features(news):
    counts = pd.Series(
        [count_abstract_entities(value, news_id)
         for news_id, value in zip(news['news_id'], news['abstract_entities_parsed'])],
        index=news.index, dtype='int32',
    )
    existing = news['num_abstract_entities'].astype('int32')
    mismatched = counts.ne(existing)
    if mismatched.any():
        raise ValueError(
            f'{int(mismatched.sum())} rows disagree with preprocessing num_abstract_entities, '
            f'e.g. {news.loc[mismatched, "news_id"].head(5).tolist()}'
        )
    return pd.DataFrame({
        'news_id': news['news_id'],
        'num_abstract_entities': counts,
        **{flag: news[flag] for flag in FLAG_COLUMNS},
    })


entity_features_by_partition = {
    partition: derive_abstract_entity_features(news) for partition, news in news_by_partition.items()
}

entity_summary = pd.DataFrame([
    {
        'partition': partition,
        'articles': len(features),
        'mean': features['num_abstract_entities'].mean(),
        'median': features['num_abstract_entities'].median(),
        'max': features['num_abstract_entities'].max(),
        'share_zero': features['num_abstract_entities'].eq(0).mean(),
        'zero_because_abstract_missing': int(
            (features['num_abstract_entities'].eq(0) & features['abstract_missing']).sum()
        ),
    }
    for partition, features in entity_features_by_partition.items()
])
display(entity_summary)
display(entity_features_by_partition['train'].head(10))

,partition,articles,mean,median,max,share_zero,zero_because_abstract_missing
0,train,43370,1.842172,1.0,30,0.271570,2210
1,validation,38421,1.792093,1.0,30,0.280237,1870
2,test,42416,1.816791,1.0,30,0.273977,2021


,news_id,num_abstract_entities,metadata_missing,abstract_missing,abstract_entities_missing
0,N10,0,False,False,False
1,N100,4,False,False,False
2,N1000,0,False,False,False
3,N10000,0,False,False,False
4,N10001,2,False,False,False
5,N10002,5,False,False,False
6,N10003,1,False,False,False
7,N10004,1,False,False,False
8,N10005,2,False,False,False
9,N10009,0,False,False,False


In [37]:
# Distribution on training articles with known metadata.
train_counts = entity_features_by_partition['train'].loc[
    ~entity_features_by_partition['train']['metadata_missing'], 'num_abstract_entities'
]
display(train_counts.value_counts().sort_index().rename('articles').to_frame().T)

num_abstract_entities,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,21,30
articles,11778,10451,8428,5644,3318,1861,1002,484,205,101,...,35,10,4,5,1,2,1,1,1,1


## 5. Title/abstract TF-IDF

Fit on training articles with metadata only, then transform every partition with the same vectorizer. Articles with missing metadata get the empty string, which becomes an all-zero row. That keeps one row for every `news_id`, so candidate and history joins stay aligned.

In [38]:
TFIDF_PARAMS = dict(
    lowercase=True,
    stop_words='english',
    token_pattern=r'(?u)\b\w\w+\b',
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    max_features=50_000,
    sublinear_tf=True,
    norm='l2',
    dtype=np.float32,
)


def title_abstract_text(news):
    text = (news['title'].str.strip() + ' ' + news['abstract'].str.strip()).str.strip()
    # Absent articles carry placeholders only; do not let them contribute tokens.
    return text.mask(news['metadata_missing'], '')


train_news = news_by_partition['train']
fit_text = title_abstract_text(train_news)[~train_news['metadata_missing']]

vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
vectorizer.fit(fit_text)
vocabulary = vectorizer.get_feature_names_out()
print(f'Fitted on {len(fit_text):,} training articles; vocabulary size {len(vocabulary):,}')

tfidf_by_partition = {}
tfidf_summary_rows = []
for partition, news in news_by_partition.items():
    matrix = vectorizer.transform(title_abstract_text(news)).astype(np.float32).tocsr()
    if matrix.shape[0] != len(news):
        raise ValueError(f'{partition}: TF-IDF rows do not match news rows.')
    tfidf_by_partition[partition] = matrix
    nnz_per_row = np.diff(matrix.indptr)
    tfidf_summary_rows.append({
        'partition': partition,
        'articles': matrix.shape[0],
        'dimensions': matrix.shape[1],
        'mean_nonzero_terms': nnz_per_row.mean(),
        'all_zero_rows': int((nnz_per_row == 0).sum()),
        'density': matrix.nnz / (matrix.shape[0] * matrix.shape[1]),
    })

tfidf_summary = pd.DataFrame(tfidf_summary_rows)
display(tfidf_summary)

Fitted on 43,370 training articles; vocabulary size 50,000


KeyboardInterrupt: 

In [ ]:
# Sanity check: highest-weighted terms for a few training articles.
def top_terms(matrix, row, k=8):
    vector = matrix.getrow(row)
    order = np.argsort(vector.data)[::-1][:k]
    return ', '.join(f'{vocabulary[vector.indices[i]]} ({vector.data[i]:.2f})' for i in order)


preview = train_news.loc[~train_news['metadata_missing']].head(5)
display(pd.DataFrame({
    'news_id': preview['news_id'].to_numpy(),
    'title': preview['title'].to_numpy(),
    'top_terms': [top_terms(tfidf_by_partition['train'], i) for i in preview.index],
}))

,news_id,title,top_terms
0,N10,7 Running Workouts for Weight Loss,"workouts (0.56), weight loss (0.50), weight (0..."
1,N100,Astros finally get timely hitting to get back ...,"loss game (0.21), prestige (0.21), meeting wed..."
2,N1000,"Sounders vs. Real Salt Lake: Highlights, stats...","brian schmetzer (0.26), sounders vs (0.26), sc..."
3,N10000,"The Day in Sports: Wednesday, October 30, 2019","sports wednesday (0.41), 30 2019 (0.40), octob..."
4,N10001,Vero Beach man charged with battery in connect...,"charged battery (0.32), vero beach (0.31), ver..."


## 6. Export

For each partition:

- `abstract_entity_features.csv`: `news_id`, `num_abstract_entities` and missingness flags
- `tfidf_title_abstract.npz`: sparse CSR matrix (rows = articles, columns = vocabulary terms)
- `tfidf_news_ids.csv`: `news_id` for each matrix row, in order

The root folder also holds the fitted `tfidf_vectorizer.pkl`, `tfidf_vocabulary.csv`, `feature_dictionary.csv` and a small `manifest.json`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for partition in PARTITIONS:
    folder = OUTPUT_DIR / partition
    folder.mkdir(parents=True, exist_ok=True)
    entity_features_by_partition[partition].to_csv(
        folder / 'abstract_entity_features.csv', index=False, encoding='utf-8'
    )
    sparse.save_npz(folder / 'tfidf_title_abstract.npz', tfidf_by_partition[partition])
    news_by_partition[partition][['news_id']].to_csv(
        folder / 'tfidf_news_ids.csv', index=False, encoding='utf-8'
    )

with (OUTPUT_DIR / 'tfidf_vectorizer.pkl').open('wb') as stream:
    pickle.dump(vectorizer, stream)
pd.DataFrame({'column': np.arange(len(vocabulary)), 'term': vocabulary}).to_csv(
    OUTPUT_DIR / 'tfidf_vocabulary.csv', index=False, encoding='utf-8'
)

feature_dictionary = pd.DataFrame([
    {'feature': 'num_abstract_entities', 'source': 'News',
     'description': 'Number of identified entities in abstract (annotation entries, not mentions)',
     'data_type': 'Count (int32)', 'file': '<partition>/abstract_entity_features.csv'},
    {'feature': 'tfidf_title_abstract', 'source': 'News text',
     'description': 'Sparse textual representation of article content (title + abstract TF-IDF)',
     'data_type': f'Numeric vector (sparse float32, {len(vocabulary)} dims)',
     'file': '<partition>/tfidf_title_abstract.npz + tfidf_news_ids.csv'},
])
feature_dictionary.to_csv(OUTPUT_DIR / 'feature_dictionary.csv', index=False, encoding='utf-8')
display(feature_dictionary)

params = {k: (v.__name__ if isinstance(v, type) else v) for k, v in TFIDF_PARAMS.items()}
params['dtype'] = 'float32'
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps({
    'source_processed_directory': str(PROCESSED_DIR),
    'source_dataset_version': source_manifest.get('dataset_version'),
    'partitions': PARTITIONS,
    'tfidf_fitted_on': 'train/news.csv excluding metadata_missing rows',
    'tfidf_fit_articles': int(len(fit_text)),
    'tfidf_vocabulary_size': int(len(vocabulary)),
    'tfidf_params': params,
}, indent=2, default=list), encoding='utf-8')
print('Saved features to', OUTPUT_DIR)

,feature,source,description,data_type,file
0,num_abstract_entities,News,Number of identified entities in abstract (ann...,Count (int32),<partition>/abstract_entity_features.csv
1,tfidf_title_abstract,News text,Sparse textual representation of article conte...,"Numeric vector (sparse float32, 50000 dims)",<partition>/tfidf_title_abstract.npz + tfidf_n...


Saved features to /Users/boogywumpy/Desktop/bt4222/outputs/features/text_entity_v1


## 7. Using the features in a model

Both features are per article, so join them to candidates (and history items) on `news_id`:

```python
from scipy import sparse
import pandas as pd

folder = OUTPUT_DIR / 'train'
entities = pd.read_csv(folder / 'abstract_entity_features.csv', dtype={'news_id': 'string'})
candidates = candidates.merge(entities[['news_id', 'num_abstract_entities']],
                              on='news_id', how='left', validate='many_to_one')

tfidf = sparse.load_npz(folder / 'tfidf_title_abstract.npz')
row_of = pd.read_csv(folder / 'tfidf_news_ids.csv', dtype='string')['news_id']
row_of = pd.Series(range(len(row_of)), index=row_of)
X_text = tfidf[row_of.loc[candidates['news_id']].to_numpy()]  # one sparse row per candidate
```

Notes:

- `num_abstract_entities` is a skewed count; consider `log1p` or a scaler **fitted on train only**.
- TF-IDF rows are already L2-normalised. Keep them sparse for linear models. For an MLP, either feed them through a sparse-aware layer or reduce them with `TruncatedSVD` fitted on train only.
- User–article similarity: average the TF-IDF rows of a user's history articles and take the cosine similarity with each candidate row. That gives a single dense feature.
- Never refit the vectorizer on validation/test text; reload `tfidf_vectorizer.pkl` if you need to transform new articles.

## To find out if a user tends to follow particular people, organisations or places, and whether a new article features ones they already care about based on entity types and behaviours on user history

## 8. Entity-type matching: person, organisation and place

Each entity annotation in `title_entities` / `abstract_entities` has an ontological `Type`. A user may follow specific **people** (celebrities, politicians) closely but have no allegiance to particular **places**. One pooled entity-overlap score would blur those signals together, so we score each type separately.

| MIND `Type` | Feature prefix | Train articles with ≥1 such entity |
|---|---|---|
| `P` | `person` | ~32% |
| `O` | `organisation` | ~40% |
| `G` (geographic) | `place` | ~38% |

Other types (concepts, works, events, …) are rarer and are left out here.

**Step 1: type-specific article representations** (one row per `news_id`, per partition)

- `{type}_entity_ids`: distinct Wikidata IDs of that type in title ∪ abstract, in first-occurrence order. No confidence threshold.
- `num_{type}_entities` and the flag `{type}_entities_missing` (article has no entity of that type).
- A 100-d **type embedding**: the mean of the pretrained TransE vectors (`entity_embedding.vec`) of those entities, L2-normalised. Train/validation use `MINDsmall_train` vectors and test uses `MINDsmall_dev`, matching where each partition's metadata came from. The two files give identical vectors for shared entities, and coverage of P/O/G mentions is 99.8%.

**Step 2: history profile** (one per impression `event_key`, built only from the *supplied* `history_ids` in `behaviors.tsv`)

- History entity set: every type-`T` entity found in any history article.
- History type embedding: the mean of the history articles' type-`T` embeddings, L2-normalised.

**Step 3: candidate matching scores** (one row per candidate exposure), for each type `T`:

| Column | Definition | Range |
|---|---|---|
| `{type}_entity_overlap` | Share of the candidate's distinct type-`T` entities that appear in the user's history | 0–1 |
| `{type}_embedding_similarity` | Cosine similarity between the candidate's type-`T` embedding and the history type-`T` embedding | −1–1 |
| `candidate_{type}_missing` | Candidate has no type-`T` entity | bool |
| `history_{type}_missing` | No history article has a type-`T` entity (includes empty histories) | bool |

When either flag is set, both scores are **0** and the flags record why, so the model can tell "no shared people" apart from "no people to compare". Overlap catches exact following ("this user reads about *this* politician"). Embedding similarity catches related entities ("other players from the same team").

No labels are used to build these features. Only the optional diagnostic in Section 11 reads training labels.

## 9. Type-specific article representations

A profile card for each article:

For every article, the notebook lists:

1. the people it mentions (e.g. Taylor Swift, Joe Biden)

2. the organisations it mentions (e.g. Apple, the Houston Astros)

3. the places it mentions (e.g. Texas, London)

This gives each. group a meaning fingerprint e.g. embedding.



In [ ]:
ENTITY_TYPES = {'person': 'P', 'organisation': 'O', 'place': 'G'}
EMBEDDING_SOURCE = {'train': 'MINDsmall_train', 'validation': 'MINDsmall_train', 'test': 'MINDsmall_dev'}
RAW_DIR = Path(os.environ.get('MIND_RAW_DIR', str(
    PROJECT_ROOT / 'data/raw' if PROJECT_ROOT is not None else PROCESSED_DIR.parents[1] / 'raw'
))).expanduser().resolve()
CANDIDATE_CHUNK_SIZE = 500_000


def load_entity_embeddings(source):
    path = RAW_DIR / source / 'entity_embedding.vec'
    if not path.is_file():
        raise FileNotFoundError(f'{path} is missing; set MIND_RAW_DIR to the raw MIND folder.')
    ids, rows = [], []
    with path.open(encoding='utf-8') as stream:
        for line in stream:
            parts = line.split()
            ids.append(parts[0])
            rows.append(np.asarray(parts[1:], dtype=np.float32))
    index = pd.Index(ids)
    if not index.is_unique:
        raise ValueError(f'{path}: duplicate entity IDs')
    return index, np.vstack(rows)


def l2_normalise(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.divide(matrix, norms, out=np.zeros_like(matrix), where=norms > 0)


def build_article_type_representations(news, embedding_index, embedding_matrix):
    # Returns the article table plus, per type, a binary article x entity matrix and an embedding matrix.
    entities = [
        json.loads(title) + json.loads(abstract)
        for title, abstract in zip(news['title_entities_parsed'], news['abstract_entities_parsed'])
    ]
    table = {'news_id': news['news_id'].to_numpy()}
    incidence, embeddings = {}, {}
    for name, code in ENTITY_TYPES.items():
        ids = [list(dict.fromkeys(
            e['WikidataId'] for e in row if e.get('Type') == code and e.get('WikidataId')
        )) for row in entities]
        vocabulary = pd.Index(sorted({i for row in ids for i in row}))
        lengths = np.array([len(row) for row in ids])
        matrix = sparse.csr_matrix(
            (np.ones(lengths.sum(), dtype=np.float32),
             (np.repeat(np.arange(len(ids)), lengths), vocabulary.get_indexer([i for row in ids for i in row]))),
            shape=(len(ids), len(vocabulary)),
        )
        position = embedding_index.get_indexer(vocabulary)
        vectors = np.zeros((len(vocabulary), embedding_matrix.shape[1]), dtype=np.float32)
        vectors[position >= 0] = embedding_matrix[position[position >= 0]]

        incidence[name] = matrix
        embeddings[name] = l2_normalise(np.asarray(matrix @ vectors, dtype=np.float32))
        table[f'{name}_entity_ids'] = [json.dumps(row) for row in ids]
        table[f'num_{name}_entities'] = lengths.astype('int32')
        table[f'{name}_entities_missing'] = lengths == 0
        table[f'{name}_entities_without_embedding'] = np.asarray(
            matrix[:, np.flatnonzero(position < 0)].sum(axis=1)).ravel().astype('int32')
    return pd.DataFrame(table), incidence, embeddings

In [ ]:
embeddings_by_source = {}
article_types_by_partition, incidence_by_partition, type_embeddings_by_partition = {}, {}, {}

for partition in PARTITIONS:
    source = EMBEDDING_SOURCE[partition]
    if source not in embeddings_by_source:
        embeddings_by_source[source] = load_entity_embeddings(source)
    news = pd.read_csv(
        PROCESSED_DIR / partition / 'news.csv',
        usecols=['news_id', 'title_entities_parsed', 'abstract_entities_parsed'],
        dtype='string', keep_default_na=False, encoding='utf-8',
    )
    if not news['news_id'].equals(news_by_partition[partition]['news_id']):
        raise ValueError(f'{partition}: news row order differs from Section 3.')
    table, incidence, embeddings = build_article_type_representations(news, *embeddings_by_source[source])
    article_types_by_partition[partition] = table
    incidence_by_partition[partition] = incidence
    type_embeddings_by_partition[partition] = embeddings

article_type_summary = pd.DataFrame([
    {
        'partition': partition,
        'type': name,
        'articles_with_type': int((~table[f'{name}_entities_missing']).sum()),
        'share_with_type': (~table[f'{name}_entities_missing']).mean(),
        'distinct_entities': incidence_by_partition[partition][name].shape[1],
        'mean_entities_when_present': table.loc[~table[f'{name}_entities_missing'], f'num_{name}_entities'].mean(),
        'entities_without_embedding': int(table[f'{name}_entities_without_embedding'].sum()),
    }
    for partition, table in article_types_by_partition.items() for name in ENTITY_TYPES
])
display(article_type_summary)
display(article_types_by_partition['train'].head(5))

,partition,type,articles_with_type,share_with_type,distinct_entities,mean_entities_when_present,entities_without_embedding
0,train,person,13788,0.317916,5520,1.522193,3
1,train,organisation,17505,0.403620,3456,1.406170,117
2,train,place,16379,0.377657,3899,1.501862,10
3,validation,person,12231,0.318342,5103,1.527921,2
4,validation,organisation,14943,0.388928,3186,1.393763,97
5,validation,place,14216,0.370006,3468,1.488956,9
6,test,person,13701,0.323015,5402,1.534487,3
7,test,organisation,16981,0.400344,3207,1.406572,112
8,test,place,15823,0.373043,3728,1.496429,8


,news_id,person_entity_ids,num_person_entities,person_entities_missing,person_entities_without_embedding,organisation_entity_ids,num_organisation_entities,organisation_entities_missing,organisation_entities_without_embedding,place_entity_ids,num_place_entities,place_entities_missing,place_entities_without_embedding
0,N10,[],0,True,0,[],0,True,0,[],0,True,0
1,N100,"[""Q3186148"", ""Q981788""]",2,False,0,"[""Q848117""]",1,False,0,[],0,True,0
2,N1000,[],0,True,0,"[""Q391353"", ""Q632511""]",2,False,0,[],0,True,0
3,N10000,[],0,True,0,[],0,True,0,[],0,True,0
4,N10001,[],0,True,0,[],0,True,0,"[""Q969026""]",1,False,0


## 10. History profiles and candidate matching scores

Each impression's history becomes a sparse event × article count matrix. Multiplying it by an article's type-specific representation gives that impression's history profile for the type. 

1. Build a profile of user from their reading history. Collect entity types from articles they clicked before, and average articles fingerprints.

2. Score each article we might show them separately based on entity types. Scoring based on overlap score & similarity score. 

    overlap score: what fraction of this article people has the user read about before?

    similarity score: how close is this article's person fingerprint to the user's?

3. Add 'nothing to compare flags'. score is set to 0 if article mention no people, or user has never read anything mentioning people.

In [ ]:
def build_history_profiles(partition):
    events = pd.read_csv(
        PROCESSED_DIR / partition / 'events.csv', usecols=['event_key', 'history_ids'],
        dtype='string', keep_default_na=False, encoding='utf-8',
    )
    if not events['event_key'].is_unique:
        raise ValueError(f'{partition}: duplicate event_key values.')
    article_row = pd.Series(
        np.arange(len(article_types_by_partition[partition])),
        index=pd.Index(article_types_by_partition[partition]['news_id']),
    )
    histories = events['history_ids'].map(json.loads)
    lengths = histories.map(len).to_numpy()
    columns = article_row.reindex([h for row in histories for h in row])
    if columns.isna().any():
        raise ValueError(f'{partition}: history articles missing from news.csv, e.g. '
                         f'{columns[columns.isna()].index[:5].tolist()}')
    history = sparse.csr_matrix(
        (np.ones(lengths.sum(), dtype=np.float32),
         (np.repeat(np.arange(len(events)), lengths), columns.to_numpy(dtype=np.int64))),
        shape=(len(events), len(article_row)),
    )
    profile = {
        'event_row': pd.Series(np.arange(len(events)), index=pd.Index(events['event_key'])),
        'article_row': article_row,
        'seen': {}, 'embedding': {}, 'missing': {},
    }
    for name in ENTITY_TYPES:
        counts = (history @ incidence_by_partition[partition][name]).tocsr()
        profile['seen'][name] = (counts > 0).astype(np.float32).tocsr()
        profile['embedding'][name] = l2_normalise(
            np.asarray(history @ type_embeddings_by_partition[partition][name], dtype=np.float32))
        profile['missing'][name] = np.asarray(counts.sum(axis=1)).ravel() == 0
    return profile


def build_entity_type_match_features(candidate_batch, partition, profile):
    # Safe to call on minibatches; labels are neither required nor returned.
    keys = ['event_key', 'candidate_position', 'news_id']
    e = profile['event_row'].reindex(candidate_batch['event_key'])
    a = profile['article_row'].reindex(candidate_batch['news_id'])
    if e.isna().any() or a.isna().any():
        raise ValueError(f'{partition}: candidate rows reference unknown events or articles.')
    e, a = e.to_numpy(dtype=np.int64), a.to_numpy(dtype=np.int64)
    out = candidate_batch[keys].reset_index(drop=True)
    for name in ENTITY_TYPES:
        candidate_entities = incidence_by_partition[partition][name][a]
        n_candidate = np.asarray(candidate_entities.sum(axis=1)).ravel()
        matched = np.asarray(profile['seen'][name][e].multiply(candidate_entities).sum(axis=1)).ravel()
        candidate_missing = n_candidate == 0
        history_missing = profile['missing'][name][e]
        either_missing = candidate_missing | history_missing
        similarity = np.einsum(
            'ij,ij->i', profile['embedding'][name][e], type_embeddings_by_partition[partition][name][a])
        out[f'{name}_entity_overlap'] = np.where(
            either_missing, 0.0, matched / np.maximum(n_candidate, 1)).astype(np.float32)
        out[f'{name}_embedding_similarity'] = np.where(either_missing, 0.0, similarity).astype(np.float32)
        out[f'candidate_{name}_missing'] = candidate_missing
        out[f'history_{name}_missing'] = history_missing
    return out


ENTITY_TYPE_MODEL_COLUMNS = [
    column for name in ENTITY_TYPES for column in (
        f'{name}_entity_overlap', f'{name}_embedding_similarity',
        f'candidate_{name}_missing', f'history_{name}_missing',
    )
]

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
match_summary_rows = []

for partition in PARTITIONS:
    folder = OUTPUT_DIR / partition
    folder.mkdir(parents=True, exist_ok=True)
    profile = build_history_profiles(partition)
    target = folder / 'candidate_entity_type_features.csv'
    temporary = target.with_suffix('.csv.partial')
    totals, rows_written = None, 0
    try:
        for batch in pd.read_csv(
            PROCESSED_DIR / partition / 'candidates.csv', chunksize=CANDIDATE_CHUNK_SIZE,
            usecols=['event_key', 'candidate_position', 'news_id'],
            dtype={'event_key': 'string', 'news_id': 'string', 'candidate_position': 'int32'},
            keep_default_na=False,
        ):
            features = build_entity_type_match_features(batch, partition, profile)
            features.to_csv(temporary, mode='w' if rows_written == 0 else 'a',
                            header=rows_written == 0, index=False, float_format='%.6g')
            rows_written += len(features)
            stats = pd.concat([
                features[ENTITY_TYPE_MODEL_COLUMNS].sum().add_prefix('sum__'),
                features[[f'{n}_entity_overlap' for n in ENTITY_TYPES]].gt(0).sum().add_prefix('nonzero__'),
            ])
            totals = stats if totals is None else totals + stats
        temporary.replace(target)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    for name in ENTITY_TYPES:
        match_summary_rows.append({
            'partition': partition, 'type': name, 'candidates': rows_written,
            'share_overlap_gt_0': totals[f'nonzero__{name}_entity_overlap'] / rows_written,
            'mean_overlap': totals[f'sum__{name}_entity_overlap'] / rows_written,
            'mean_embedding_similarity': totals[f'sum__{name}_embedding_similarity'] / rows_written,
            'share_candidate_missing': totals[f'sum__candidate_{name}_missing'] / rows_written,
            'share_history_missing': totals[f'sum__history_{name}_missing'] / rows_written,
        })
    print(f'{partition}: exported {rows_written:,} candidate rows')
    del profile

match_summary = pd.DataFrame(match_summary_rows)
display(match_summary)

train: exported 3,389,342 candidate rows
validation: exported 2,454,102 candidate rows
test: exported 2,740,998 candidate rows


,partition,type,candidates,share_overlap_gt_0,mean_overlap,mean_embedding_similarity,share_candidate_missing,share_history_missing
0,train,person,3389342,0.040260,0.028734,0.180119,0.692830,0.069166
1,train,organisation,3389342,0.030342,0.024491,0.158777,0.693087,0.086995
2,train,place,3389342,0.041790,0.033680,0.182567,0.739151,0.095768
3,validation,person,2454102,0.035542,0.027775,0.177581,0.685727,0.071988
4,validation,organisation,2454102,0.031612,0.024953,0.133685,0.743898,0.094250
5,validation,place,2454102,0.049147,0.037464,0.181394,0.739493,0.099686
6,test,person,2740998,0.036258,0.026795,0.179892,0.670972,0.080536
7,test,organisation,2740998,0.024389,0.019510,0.132039,0.741314,0.098088
8,test,place,2740998,0.038378,0.028561,0.189973,0.718810,0.109363


## 11. Check: do type-specific matches relate to clicks? (train only)

A quick univariate check on **training** labels. For each type, compare click-through rate (CTR) when the score can't be computed (a missing flag), when there is no overlap, and when there is any overlap. If the person and organisation lifts differ from place, that supports keeping the scores separate rather than pooling them.

On average, users click about 4 in every 100 articles shown.
When an article mentions a person the user has read about before, they click about 1.6× as often.
For an organisation it's about 1.8×, and for a place only about 1.3×.

In [ ]:
train_features = pd.read_csv(
    OUTPUT_DIR / 'train' / 'candidate_entity_type_features.csv',
    dtype={'event_key': 'string', 'news_id': 'string', 'candidate_position': 'int32'},
)
train_labels = pd.read_csv(
    PROCESSED_DIR / 'train' / 'candidates.csv',
    usecols=['event_key', 'candidate_position', 'label'],
    dtype={'event_key': 'string', 'candidate_position': 'int32', 'label': 'int8'},
)
train_features = train_features.merge(
    train_labels, on=['event_key', 'candidate_position'], how='left', validate='one_to_one')
base_ctr = train_features['label'].mean()

diagnostic_rows = []
for name in ENTITY_TYPES:
    missing = train_features[f'candidate_{name}_missing'] | train_features[f'history_{name}_missing']
    overlap = train_features[f'{name}_entity_overlap']
    for group, mask in [('missing', missing), ('no overlap', ~missing & overlap.eq(0)),
                        ('overlap > 0', ~missing & overlap.gt(0))]:
        diagnostic_rows.append({
            'type': name, 'group': group, 'candidates': int(mask.sum()),
            'ctr': train_features.loc[mask, 'label'].mean(),
            'lift_vs_base': train_features.loc[mask, 'label'].mean() / base_ctr,
        })
    similarity = train_features.loc[~missing, f'{name}_embedding_similarity']
    diagnostic_rows.append({
        'type': name, 'group': 'similarity corr. with click (non-missing)',
        'candidates': int((~missing).sum()),
        'ctr': np.corrcoef(similarity, train_features.loc[~missing, 'label'])[0, 1],
        'lift_vs_base': np.nan,
    })

print(f'Base training CTR: {base_ctr:.4f}')
display(pd.DataFrame(diagnostic_rows))
del train_features, train_labels

Base training CTR: 0.0418


,type,group,candidates,ctr,lift_vs_base
0,person,missing,2416034,0.037881,0.906982
1,person,no overlap,836854,0.048975,1.172617
2,person,overlap > 0,136454,0.066337,1.588325
3,person,similarity corr. with click (non-missing),973308,0.026871,NaN
4,organisation,missing,2436517,0.039355,0.942281
5,organisation,no overlap,849987,0.044492,1.065289
6,organisation,overlap > 0,102838,0.076343,1.827900
7,organisation,similarity corr. with click (non-missing),952825,0.036379,NaN
8,place,missing,2587176,0.040436,0.968155
9,place,no overlap,660525,0.044357,1.062049


## 12. Export entity-type representations

For each partition (the candidate scores were already written in Section 10):

- `article_entity_types.csv`: `news_id`, per-type entity IDs (JSON), counts, missing flags and the number of entities without a pretrained vector
- `article_entity_type_embeddings.npz`: `news_id` plus one `(articles × 100)` float32 array per type (`person`, `organisation`, `place`)
- `candidate_entity_type_features.csv`: `event_key`, `candidate_position`, `news_id` and the 12 matching columns

The feature dictionary and manifest are updated with the new features.

In [ ]:
for partition in PARTITIONS:
    folder = OUTPUT_DIR / partition
    article_types_by_partition[partition].to_csv(folder / 'article_entity_types.csv', index=False, encoding='utf-8')
    np.savez_compressed(
        folder / 'article_entity_type_embeddings.npz',
        news_id=article_types_by_partition[partition]['news_id'].to_numpy(dtype=str),
        **type_embeddings_by_partition[partition],
    )

entity_type_dictionary = pd.DataFrame([
    row for name, code in ENTITY_TYPES.items() for row in [
        {'feature': f'{name}_entity_overlap', 'source': 'News (entity Type) + behaviors (history)',
         'description': f'Share of candidate {name} entities (Type {code}) seen in user history',
         'data_type': 'Numeric score (float32, 0-1)', 'file': '<partition>/candidate_entity_type_features.csv'},
        {'feature': f'{name}_embedding_similarity', 'source': 'News (entity Type) + behaviors (history)',
         'description': f'Cosine similarity of candidate and history mean {name} entity embeddings',
         'data_type': 'Numeric score (float32, -1-1)', 'file': '<partition>/candidate_entity_type_features.csv'},
        {'feature': f'candidate_{name}_missing', 'source': 'News (entity Type)',
         'description': f'Candidate article has no {name} entity; scores set to 0',
         'data_type': 'Flag (bool)', 'file': '<partition>/candidate_entity_type_features.csv'},
        {'feature': f'history_{name}_missing', 'source': 'Behaviors (history)',
         'description': f'No history article has a {name} entity (incl. empty history); scores set to 0',
         'data_type': 'Flag (bool)', 'file': '<partition>/candidate_entity_type_features.csv'},
    ]
] + [
    {'feature': 'entity_type_article_representation', 'source': 'News (entity Type)',
     'description': 'Per-type entity IDs, counts and mean TransE embedding (person/organisation/place)',
     'data_type': 'IDs + counts + 3 x 100-d vector',
     'file': '<partition>/article_entity_types.csv + article_entity_type_embeddings.npz'},
])
feature_dictionary = pd.concat([
    feature_dictionary[~feature_dictionary['feature'].isin(entity_type_dictionary['feature'])],
    entity_type_dictionary,
], ignore_index=True)
feature_dictionary.to_csv(OUTPUT_DIR / 'feature_dictionary.csv', index=False, encoding='utf-8')
display(feature_dictionary)

manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text(encoding='utf-8'))
manifest['entity_types'] = ENTITY_TYPES
manifest['entity_embedding_source'] = {p: EMBEDDING_SOURCE[p] for p in PARTITIONS}
manifest['entity_type_model_columns'] = ENTITY_TYPE_MODEL_COLUMNS
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=list), encoding='utf-8')
print('Saved entity-type features to', OUTPUT_DIR)

,feature,source,description,data_type,file
0,num_abstract_entities,News,Number of identified entities in abstract (ann...,Count (int32),<partition>/abstract_entity_features.csv
1,tfidf_title_abstract,News text,Sparse textual representation of article conte...,"Numeric vector (sparse float32, 50000 dims)",<partition>/tfidf_title_abstract.npz + tfidf_n...
2,person_entity_overlap,News (entity Type) + behaviors (history),Share of candidate person entities (Type P) se...,"Numeric score (float32, 0-1)",<partition>/candidate_entity_type_features.csv
3,person_embedding_similarity,News (entity Type) + behaviors (history),Cosine similarity of candidate and history mea...,"Numeric score (float32, -1-1)",<partition>/candidate_entity_type_features.csv
4,candidate_person_missing,News (entity Type),Candidate article has no person entity; scores...,Flag (bool),<partition>/candidate_entity_type_features.csv
5,history_person_missing,Behaviors (history),No history article has a person entity (incl. ...,Flag (bool),<partition>/candidate_entity_type_features.csv
6,organisation_entity_overlap,News (entity Type) + behaviors (history),Share of candidate organisation entities (Type...,"Numeric score (float32, 0-1)",<partition>/candidate_entity_type_features.csv
7,organisation_embedding_similarity,News (entity Type) + behaviors (history),Cosine similarity of candidate and history mea...,"Numeric score (float32, -1-1)",<partition>/candidate_entity_type_features.csv
8,candidate_organisation_missing,News (entity Type),Candidate article has no organisation entity; ...,Flag (bool),<partition>/candidate_entity_type_features.csv
9,history_organisation_missing,Behaviors (history),No history article has a organisation entity (...,Flag (bool),<partition>/candidate_entity_type_features.csv


Saved entity-type features to /Users/boogywumpy/Desktop/bt4222/outputs/features/text_entity_v1


## 13. Using the entity-type features in a model

The candidate scores are keyed by `(event_key, candidate_position)`, so join them to the candidate table one-to-one:

```python
entity_type = pd.read_csv(OUTPUT_DIR / 'train' / 'candidate_entity_type_features.csv',
                          dtype={'event_key': 'string', 'news_id': 'string'})
candidates = candidates.merge(entity_type.drop(columns='news_id'),
                              on=['event_key', 'candidate_position'], how='left', validate='one_to_one')
X_entity_type = candidates[ENTITY_TYPE_MODEL_COLUMNS].to_numpy(dtype='float32')
```

Alternatively, call `build_entity_type_match_features(candidate_batch, partition, build_history_profiles(partition))` on model minibatches inside this notebook.

- Always feed the missing flags **with** the scores. A 0 score is ambiguous without them.
- The scores are already bounded, so no scaler is needed.
- For a neural model, the per-type article embeddings in `article_entity_type_embeddings.npz` can also feed a candidate-attention user encoder directly, with one attention head per type.